# 02. Bayesian Synthetic Control for a Marketing Pilot

This notebook is a self-contained introduction to Bayesian synthetic control methods using a marketing example.

It follows the spirit of Kim, Lee, and Gupta (2020), *Bayesian Synthetic Control Methods*:
- the treated unit is small in number, often a single aggregate unit,
- identification comes from matching the treated unit to untreated donors primarily on the outcome path,
- a long pre-treatment panel is valuable, and
- Bayesian estimation provides posterior uncertainty around the counterfactual and the treatment effect.

For instructional clarity, this notebook uses a simple conjugate Bayesian regression representation of Bayesian synthetic control. That is not a full replication of every modeling choice in the paper, but it preserves the key ideas: flexible weights, shrinkage, and posterior inference.

## Learning Goals

By the end of this notebook, you should be able to:
- Explain when Bayesian synthetic control is a good design choice.
- Define the source of the shock, the treated unit, and the donor pool.
- State the identification assumptions behind the design.
- Compare classical synthetic control to a Bayesian alternative.
- Estimate a posterior distribution for the counterfactual path.
- Interpret posterior treatment effects and donor weights.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")
rng = np.random.default_rng(2031)

## Marketing Setup And Source Of The Shock

Suppose a retailer launches an AI-driven retention engine in **Austin** beginning in month 25. The new system decides which churn-risk customers receive high-value win-back offers and personalized creative.

Why Austin first?
- Austin is the first market where the CRM and point-of-sale systems were fully integrated.
- The local marketing team was selected as the pilot team for the new workflow.
- Other markets remain untreated during the sample window.

That creates a clearly timed market-level shock. The design problem is to estimate what Austin would have looked like after month 25 if the retention engine had not launched.

## When This Method Is Useful

Bayesian synthetic control is especially natural when:
- you are matching treated and control units primarily on the outcome path,
- the number of treated units is small,
- you have a sufficiently long pre-treatment panel, and
- you want posterior uncertainty instead of a single synthetic control path.

Relative to the classical synthetic control notebook, the Bayesian version is useful when you want more flexible donor coefficients and a transparent posterior distribution over the counterfactual.

In [ ]:
months = np.arange(1, 31)
treatment_start = 25
pre_months = months[months < treatment_start]
post_months = months[months >= treatment_start]

treated_market = "Austin"
donor_markets = [f"Market_{idx:02d}" for idx in range(1, 13)]

market_chars = pd.DataFrame({
    "market_id": donor_markets,
    "population_index": rng.normal(100, 12, len(donor_markets)),
    "digital_share": np.clip(rng.normal(0.41, 0.06, len(donor_markets)), 0.22, 0.68),
    "promo_intensity": rng.normal(1.0, 0.18, len(donor_markets)),
    "latent_loading_1": rng.normal(1.0, 0.20, len(donor_markets)),
    "latent_loading_2": rng.normal(0.0, 0.55, len(donor_markets)),
    "market_intercept": rng.normal(0, 5, len(donor_markets)),
})

latent_factor_1 = np.array([-10, -9, -8, -7, -5, -4, -2, -1, 0, 2, 3, 5, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25])
latent_factor_2 = np.array([-6, -5, -3, -1, 1, 3, 5, 6, 7, 8, 9, 9, 8, 7, 6, 5, 3, 1, 0, -1, -2, -2, -1, 0, 1, 2, 3, 3, 4, 4])
seasonality = np.array([-2, -1, 0, 1, 2, 2, 1, 0, -1, -1, 0, 1] * 2 + [-2, -1, 0, 1, 2, 2])

donor_outcomes = {}
for row in market_chars.itertuples(index=False):
    baseline = 170 + 0.80 * row.population_index + 45 * row.digital_share + 8 * row.promo_intensity + row.market_intercept
    donor_outcomes[row.market_id] = (
        baseline
        + row.latent_loading_1 * latent_factor_1
        + row.latent_loading_2 * latent_factor_2
        + seasonality
        + rng.normal(0, 1.5, len(months))
    )

donor_outcome_wide = pd.DataFrame(donor_outcomes, index=months)

true_bayesian_weights = pd.Series(0.0, index=donor_markets)
true_bayesian_weights.loc[["Market_02", "Market_05", "Market_09", "Market_11"]] = [0.80, 0.40, -0.30, 0.25]

untreated_treated_path = 20 + donor_outcome_wide.dot(true_bayesian_weights) + rng.normal(0, 0.9, len(months))
treatment_effect_path = np.array([0] * len(pre_months) + [12, 14, 16, 18, 19, 20])
treated_outcome = untreated_treated_path + treatment_effect_path + rng.normal(0, 1.2, len(months))

panel_rows = []
for market_id in donor_markets:
    for month in months:
        panel_rows.append(
            {
                "market_id": market_id,
                "month": month,
                "orders": donor_outcome_wide.loc[month, market_id],
                "treated_market": 0,
            }
        )

for month in months:
    panel_rows.append(
        {
            "market_id": treated_market,
            "month": month,
            "orders": treated_outcome.loc[month],
            "treated_market": 1,
        }
    )

panel = pd.DataFrame(panel_rows)
panel.head()

## Identification Assumptions

The core assumptions are close to classical synthetic control, with Bayesian estimation changing the way the counterfactual is learned and summarized:
- **No spillovers:** the Austin pilot does not affect donor markets.
- **No Austin-only confound at month 25:** nothing else turns on only in Austin at the same time.
- **Stable untreated relationship:** the pre-treatment relationship between Austin and the donor pool remains informative after treatment.
- **A long enough pre-treatment panel:** outcome matching is only credible if pre-treatment dynamics are observed in enough detail.
- **Priors regularize rather than identify the design:** the Bayesian prior helps stabilize estimation, but it does not rescue a bad design.

The empirical question is therefore whether donor markets can reproduce Austin well before treatment, and whether the posterior counterfactual diverges meaningfully only after the shock.

## 1. Inspect The Outcome Path Before Modeling

Because this method is outcome-path based, start by looking at Austin against the raw donor average. This is only a baseline visual. The actual estimator will use donor-specific weights.

In [ ]:
outcome_wide = panel.pivot(index="month", columns="market_id", values="orders").sort_index()
treated_path = outcome_wide[treated_market]
raw_donor_average = outcome_wide[donor_markets].mean(axis=1)

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(treated_path.index, treated_path, color="tab:blue", linewidth=3, marker="o", label="Austin")
ax.plot(raw_donor_average.index, raw_donor_average, color="tab:gray", linewidth=2.5, marker="o", label="Raw donor average")
ax.axvline(treatment_start - 0.5, color="black", linestyle="--", linewidth=1.5)
ax.set_title("Austin versus the raw donor average")
ax.set_xlabel("Month")
ax.set_ylabel("Orders")
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.legend(frameon=True)
plt.show()

## 2. Fit A Classical Synthetic Control Benchmark

Before fitting the Bayesian model, it is useful to build a classical synthetic control benchmark. That benchmark constrains donor weights to be nonnegative and to sum to one.

In [ ]:
def project_to_simplex(values):
    values = np.asarray(values, dtype=float)
    sorted_values = np.sort(values)[::-1]
    cumulative = np.cumsum(sorted_values)
    rho = np.where(sorted_values * np.arange(1, len(values) + 1) > (cumulative - 1))[0][-1]
    theta = (cumulative[rho] - 1) / (rho + 1)
    return np.maximum(values - theta, 0)

def fit_classical_scm(target, donor_matrix, n_iter=20000, learning_rate=0.08):
    target = np.asarray(target, dtype=float)
    donor_matrix = np.asarray(donor_matrix, dtype=float)
    weights = np.ones(donor_matrix.shape[1]) / donor_matrix.shape[1]
    for step in range(n_iter):
        gradient = 2 * donor_matrix.T @ (donor_matrix @ weights - target) / donor_matrix.shape[0]
        step_size = learning_rate / np.sqrt(step + 1)
        weights = project_to_simplex(weights - step_size * gradient)
    return weights

y_pre = treated_path.loc[pre_months].to_numpy()
X_pre = outcome_wide.loc[pre_months, donor_markets].to_numpy()

classical_weights = pd.Series(
    fit_classical_scm(y_pre, X_pre),
    index=donor_markets,
    name="classical_weight",
)

classical_counterfactual = pd.Series(
    outcome_wide[donor_markets].to_numpy() @ classical_weights.to_numpy(),
    index=months,
    name="classical_counterfactual",
)

classical_weights[classical_weights > 0.01].sort_values(ascending=False).to_frame().round(3)

## 3. Fit Bayesian Synthetic Control

Now fit a Bayesian model for Austin's pre-treatment path using the donor outcomes as predictors:

\[
Y^{Austin}_t = \alpha + X_t \beta + \varepsilon_t
\]

with shrinkage coming from a prior on the coefficients. The paper uses richer Bayesian machinery and MCMC. Here we use a conjugate Bayesian linear model so the core logic stays transparent:
- donor coefficients are not forced to be nonnegative,
- they do not have to sum to one, and
- posterior draws give a distribution for the counterfactual and the treatment effect.

In [ ]:
X_all = outcome_wide[donor_markets].to_numpy()
Z_pre = np.column_stack([np.ones(len(pre_months)), X_pre])
Z_all = np.column_stack([np.ones(len(months)), X_all])

p = len(donor_markets)
theta0 = np.zeros(p + 1)
V0 = np.diag([1_000.0] + [1.0] * p)
V0_inv = np.linalg.inv(V0)
a0 = 2.0
b0 = 20.0

precision_n = V0_inv + Z_pre.T @ Z_pre
V_n = np.linalg.inv(precision_n)
theta_n = V_n @ (V0_inv @ theta0 + Z_pre.T @ y_pre)
a_n = a0 + len(y_pre) / 2
b_n = b0 + 0.5 * (y_pre @ y_pre + theta0 @ V0_inv @ theta0 - theta_n @ precision_n @ theta_n)

n_draws = 4_000
sigma2_draws = 1 / rng.gamma(shape=a_n, scale=1 / b_n, size=n_draws)
theta_draws = np.vstack(
    [rng.multivariate_normal(theta_n, sigma2 * V_n) for sigma2 in sigma2_draws]
)

bayes_counterfactual_draws = theta_draws @ Z_all.T
bayes_counterfactual_mean = bayes_counterfactual_draws.mean(axis=0)
bayes_counterfactual_lower = np.quantile(bayes_counterfactual_draws, 0.05, axis=0)
bayes_counterfactual_upper = np.quantile(bayes_counterfactual_draws, 0.95, axis=0)

bayes_gap_draws = treated_path.to_numpy()[None, :] - bayes_counterfactual_draws
bayes_gap_mean = bayes_gap_draws.mean(axis=0)
bayes_gap_lower = np.quantile(bayes_gap_draws, 0.05, axis=0)
bayes_gap_upper = np.quantile(bayes_gap_draws, 0.95, axis=0)

posterior_mean_weights = pd.Series(theta_n[1:], index=donor_markets, name="posterior_mean_weight")
posterior_mean_weights.sort_values(key=lambda series: series.abs(), ascending=False).head(8).to_frame().round(3)

## 4. Compare Pre-Treatment Fit

The first design question is whether the Bayesian counterfactual improves the pre-treatment fit relative to simpler alternatives. Here we compare three objects:
- the raw donor average,
- the classical constrained synthetic control, and
- the posterior mean counterfactual from the Bayesian model.

In [ ]:
bayes_counterfactual_series = pd.Series(bayes_counterfactual_mean, index=months, name="bayes_counterfactual")

rmspe_table = pd.DataFrame(
    {
        "pre_rmspe": [
            np.sqrt(np.mean((treated_path.loc[pre_months] - raw_donor_average.loc[pre_months]) ** 2)),
            np.sqrt(np.mean((treated_path.loc[pre_months] - classical_counterfactual.loc[pre_months]) ** 2)),
            np.sqrt(np.mean((treated_path.loc[pre_months] - bayes_counterfactual_series.loc[pre_months]) ** 2)),
        ]
    },
    index=["Raw donor average", "Classical synthetic control", "Bayesian synthetic control"],
)
rmspe_table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 6))
ax.plot(treated_path.index, treated_path, color="tab:blue", linewidth=3, marker="o", label="Austin")
ax.plot(raw_donor_average.index, raw_donor_average, color="tab:gray", linewidth=2.25, label="Raw donor average")
ax.plot(classical_counterfactual.index, classical_counterfactual, color="tab:green", linewidth=2.5, label="Classical synthetic control")
ax.plot(months, bayes_counterfactual_mean, color="tab:orange", linewidth=2.75, label="Bayesian posterior mean counterfactual")
ax.fill_between(months, bayes_counterfactual_lower, bayes_counterfactual_upper, color="tab:orange", alpha=0.18, label="Bayesian 90% credible band")
ax.axvline(treatment_start - 0.5, color="black", linestyle="--", linewidth=1.5)
ax.set_title("Treated path versus classical and Bayesian counterfactuals")
ax.set_xlabel("Month")
ax.set_ylabel("Orders")
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.legend(frameon=True, loc="upper left")
plt.show()

## 5. Estimate The Posterior Treatment Effect

Once treatment begins, the treatment effect is the difference between Austin and the posterior counterfactual path. The Bayesian version gives an entire posterior distribution for that gap in each post-treatment month and for the average post-treatment effect.

In [ ]:
post_index = months >= treatment_start
average_post_effect_draws = bayes_gap_draws[:, post_index].mean(axis=1)

effect_summary = pd.Series(
    {
        "Posterior mean average post-treatment effect": average_post_effect_draws.mean(),
        "5th percentile": np.quantile(average_post_effect_draws, 0.05),
        "95th percentile": np.quantile(average_post_effect_draws, 0.95),
        "Probability effect > 0": (average_post_effect_draws > 0).mean(),
        "True average effect": treatment_effect_path[len(pre_months):].mean(),
    }
)
effect_summary.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(months, bayes_gap_mean, color="tab:red", linewidth=2.75, marker="o", label="Posterior mean gap")
ax.fill_between(months, bayes_gap_lower, bayes_gap_upper, color="tab:red", alpha=0.18, label="90% credible band")
ax.axhline(0, color="black", linewidth=1)
ax.axvline(treatment_start - 0.5, color="black", linestyle="--", linewidth=1.5)
ax.set_title("Posterior treatment effect by month")
ax.set_xlabel("Month")
ax.set_ylabel("Austin minus counterfactual")
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.legend(frameon=True)
plt.show()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(average_post_effect_draws, bins=35, color="tab:purple", alpha=0.75, edgecolor="white")
ax.axvline(average_post_effect_draws.mean(), color="black", linestyle="--", linewidth=1.5)
ax.set_title("Posterior distribution of the average post-treatment effect")
ax.set_xlabel("Average post-treatment effect")
ax.set_ylabel("Posterior draws")
plt.show()

## 6. What Do The Bayesian Weights Tell Us?

Because this is simulated data, we can compare the true donor coefficients used to construct Austin's untreated path with the classical synthetic control weights and the Bayesian posterior coefficients.

Two important differences from classical synthetic control:
- Bayesian coefficients are not forced to be nonnegative.
- They do not have to sum to one.

That flexibility is one of the key ideas in the paper.

In [ ]:
weight_summary = pd.DataFrame(
    {
        "true_weight": true_bayesian_weights,
        "classical_weight": classical_weights,
        "bayes_mean": theta_draws[:, 1:].mean(axis=0),
        "bayes_p05": np.quantile(theta_draws[:, 1:], 0.05, axis=0),
        "bayes_p95": np.quantile(theta_draws[:, 1:], 0.95, axis=0),
    },
    index=donor_markets,
)
weight_summary[weight_summary[["true_weight", "classical_weight", "bayes_mean"]].abs().max(axis=1) > 0.05].sort_values("bayes_mean", key=lambda series: series.abs(), ascending=False).round(3)

## Takeaways

- Bayesian synthetic control is useful when the number of treated units is small and the design relies on matching the outcome path over a long pre-treatment window.
- The shock here is the month-25 launch of an AI-driven retention engine in Austin.
- The identifying logic still depends on no spillovers, no Austin-only confound at the intervention date, and credible pre-treatment fit.
- The classical synthetic control benchmark is useful, but the Bayesian version allows more flexible donor coefficients and yields posterior uncertainty directly.
- In this simulation, the Bayesian model improves pre-treatment fit and produces a posterior distribution for the post-treatment effect rather than a single gap line.

In [ ]:
summary = pd.Series(
    {
        "Treated market": treated_market,
        "Donor markets": len(donor_markets),
        "Pre-treatment months": len(pre_months),
        "Post-treatment months": len(post_months),
        "Classical SCM pre-RMSPE": round(rmspe_table.loc["Classical synthetic control", "pre_rmspe"], 2),
        "Bayesian SCM pre-RMSPE": round(rmspe_table.loc["Bayesian synthetic control", "pre_rmspe"], 2),
        "Posterior mean average effect": round(average_post_effect_draws.mean(), 2),
        "True average effect": round(treatment_effect_path[len(pre_months):].mean(), 2),
    }
)
summary